# NYC Restaurant Intelligence Platform
## Notebook 04: Database Schema and Load Preparation

**Input**: the analysis tables in `data/cleaned/`
**Output**: normalised tables in `data/processed/` plus `sql/schema.sql`

### Why the analysis tables are not the database tables

`data/cleaned/` is shaped for pandas: each table carries every column an analyst might want,
so a single `read_csv` answers most questions without joins. That convenience is paid for in
repetition — `violations.csv` stores the restaurant's name, borough and cuisine on all
288,486 rows, though there are only 31,222 restaurants.

A database is shaped the opposite way. Repeated values move into their own tables and rows
point at them by id. Section 2 measures what that is worth here; the short version is
**250 MB of in-memory data becomes about 25 MB**, which also keeps the whole database inside
Supabase's 500 MB free tier.

### Target schema

Two fact tables carrying the events, two reference tables carrying the things they refer to.

```
   neighborhoods            violation_codes
   (195 rows)               (241 rows)
   nta_code  PK             violation_id  PK
        ^                        ^
        |                        |
   restaurants  <---- inspections <---- violations
   (31,222)             (93,106)         (288,486)
   camis  PK            inspection_id PK  violation_row_id PK
                        camis  FK         inspection_id FK
                                          violation_id   FK
```

Read it as a sentence: **a restaurant sits in a neighbourhood, has many inspections, and each
inspection cites many violations, each of which is one of 241 known violation types.**

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)

PROJ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEANED = PROJ / "data" / "cleaned"
PROCESSED = PROJ / "data" / "processed"
RAW = PROJ / "data" / "raw"
SQL = PROJ / "sql"
PROCESSED.mkdir(parents=True, exist_ok=True)
SQL.mkdir(exist_ok=True)

restaurants_raw = pd.read_csv(CLEANED / "restaurants.csv")
inspections_raw = pd.read_csv(CLEANED / "inspections.csv")
violations_raw = pd.read_csv(CLEANED / "violations.csv")
boundaries_raw = pd.read_csv(CLEANED / "borough_boundaries.csv")
nta_raw = pd.read_csv(RAW / "nta_population_2010.csv").query("year == 2010")


def mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / 1024**2


before = {
    "restaurants": mb(restaurants_raw),
    "inspections": mb(inspections_raw),
    "violations": mb(violations_raw),
}
for name, size in before.items():
    print(f"{name:14} {len(globals()[name + '_raw']):>7,} rows   {size:6.1f} MB")
print(f"{'total':14} {sum(before.values()):>21.1f} MB")

restaurants     31,222 rows     24.1 MB
inspections     93,106 rows     46.0 MB
violations     288,486 rows    251.8 MB
total                          321.9 MB


## 1. Reference tables

These hold the values that the fact tables would otherwise repeat.

### 1.1 `neighborhoods` — 195 rows

In [2]:
neighborhoods = (
    nta_raw.rename(columns={"nta_code": "nta_code", "nta_name": "nta_name",
                            "borough": "borough", "population": "population_2010"})
    [["nta_code", "nta_name", "borough", "population_2010"]]
    .drop_duplicates("nta_code")
    .sort_values("nta_code")
    .reset_index(drop=True)
)
print(f"neighborhoods: {len(neighborhoods)} rows")
neighborhoods.head(3)

neighborhoods: 195 rows


,nta_code,nta_name,borough,population_2010
0,BK09,Brooklyn Heights-Cobble Hill,Brooklyn,22887
1,BK17,Sheepshead Bay-Gerritsen Beach-Manhattn Bch,Brooklyn,64518
2,BK19,Brighton Beach,Brooklyn,35547


### 1.2 `violation_codes` — the biggest single win

`violation_description` alone occupies 67.7 MB in the analysis table. There are only a couple
of hundred distinct texts; every other occurrence is a copy.

Note that code and description are **not** one-to-one — DOHMH has reworded some violations
over the years, so a single code can appear with several descriptions. The lookup is keyed on
the distinct *pair*, which preserves the original wording of every record rather than
silently collapsing revisions together.

In [3]:
pairs = (
    violations_raw[["violation_code", "violation_description"]]
    .drop_duplicates()
    .sort_values(["violation_code", "violation_description"])
    .reset_index(drop=True)
)
violation_codes = pairs.assign(violation_id=np.arange(1, len(pairs) + 1))[
    ["violation_id", "violation_code", "violation_description"]
]

n_codes = violation_codes["violation_code"].nunique()
print(f"violation_codes: {len(violation_codes)} rows covering {n_codes} distinct codes")
print(f"Codes carrying more than one wording: "
      f"{(violation_codes.groupby('violation_code').size() > 1).sum()}")
violation_codes.head(3)

violation_codes: 241 rows covering 149 distinct codes
Codes carrying more than one wording: 59


,violation_id,violation_code,violation_description
0,1,02A,Food not cooked to required minimum temperature.
1,2,02A,Time/Temperature Control for Safety (TCS) food...
2,3,02A,Time/Temperature Control for Safety (TCS) food...


### 1.3 `borough_boundaries` — stays out of the database

Borough outlines are needed for maps, and the obvious move is to store them alongside
everything else. Measuring the file first says otherwise:

| Borough | Length of the geometry string |
|---|---|
| Queens | 1,373,139 characters |
| Brooklyn | 795,546 |
| Staten Island | 341,911 |
| Bronx | 341,885 |
| Manhattan | 233,138 |

Five rows, but a single cell holding 1.3 MB of coordinates. Browser-based CSV importers
cannot handle a field that size — and more to the point, **this is not what a database is
for**. The other five tables get filtered, joined and aggregated on every dashboard
interaction. Borough outlines are never queried: they are read once, whole, to draw a map,
and New York's borders do not change. That makes them a static asset, closer to an image
than to a record.

They stay in the repository as a file, and are the one thing under `data/` that git tracks.

In [4]:
# Kept only to document why this file is excluded — the sizes make the case
geometry_size = (
    boundaries_raw.assign(wkt_chars=boundaries_raw["geometry_wkt"].str.len())
    .sort_values("wkt_chars", ascending=False)[["boro_name", "wkt_chars"]]
    .reset_index(drop=True)
)
display(geometry_size)
print(f"Largest single cell: {geometry_size['wkt_chars'].max() / 1024**2:.1f} MB of text")
print("Not loaded into Postgres. It stays in the repository as")
print("data/cleaned/borough_boundaries.csv, ready for the notebook maps and for any")
print("future dashboard layer that needs true borough outlines rather than a basemap.")

,boro_name,wkt_chars
0,Queens,1373139
1,Brooklyn,795546
2,Staten Island,341911
3,Bronx,341885
4,Manhattan,233138


Largest single cell: 1.3 MB of text
Not loaded into Postgres. It stays in the repository as
data/cleaned/borough_boundaries.csv, ready for the notebook maps and for any
future dashboard layer that needs true borough outlines rather than a basemap.


## 2. Fact tables

Each keeps its own measurements and replaces everything else with a foreign key.

### 2.1 `restaurants` — 31,222 rows

In [5]:
RESTAURANT_COLS = [
    "camis", "dba", "boro", "building", "street", "zipcode", "phone",
    "cuisine", "latitude", "longitude", "nta", "community_board", "council_district",
    # Pre-computed roll-ups. Strictly these are derivable from inspections, but the
    # dashboard reads them on every page load, so they are stored rather than recomputed.
    "never_inspected", "n_inspections", "first_inspection", "last_inspection",
    "avg_score", "latest_score", "latest_grade", "total_violations", "total_critical",
    "ever_closed",
]

restaurants = (
    restaurants_raw[RESTAURANT_COLS]
    .rename(columns={"nta": "nta_code"})
    .copy()
)
restaurants["ever_closed"] = restaurants["ever_closed"].fillna(False).astype(bool)

print(f"restaurants: {len(restaurants):,} rows, {mb(restaurants):.1f} MB")
print(f"  dropped from the analysis table: "
      f"{sorted(set(restaurants_raw.columns) - set(RESTAURANT_COLS))}")
restaurants.head(3)

restaurants: 31,222 rows, 19.5 MB
  dropped from the analysis table: ['dba_raw', 'latest_grade_date']


,camis,dba,boro,building,street,zipcode,phone,cuisine,latitude,longitude,nta_code,community_board,council_district,never_inspected,n_inspections,first_inspection,last_inspection,avg_score,latest_score,latest_grade,total_violations,total_critical,ever_closed
0,30075445,MORRIS PARK BAKE SHOP,Bronx,1007,MORRIS PARK AVENUE,10462.0,7188924968,BAKERY PRODUCTS/DESSERTS,40.848231,-73.855972,BX37,211.0,13.0,False,4.0,2023-08-01,2026-02-27,16.75,7.0,A,11.0,5.0,False
1,30191841,D.J. REYNOLDS,Manhattan,351,WEST 57 STREET,10019.0,2122452912,IRISH,40.767326,-73.984310,MN15,104.0,3.0,False,3.0,2024-11-20,2026-07-30,15.67,13.0,A,11.0,6.0,False
2,40356078,"I. & L. DELICACIES,INC.",Manhattan,1500-2N,D AVE.,NaN,NaN,UNKNOWN,NaN,NaN,NaN,NaN,NaN,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False


### 2.2 `inspections` — 93,106 rows

In [6]:
INSPECTION_COLS = [
    "inspection_id", "camis", "inspection_date", "inspection_type",
    "inspection_category", "inspection_stage", "score", "grade", "grade_date",
    "is_gradeable", "n_violations", "n_critical", "closed_by_dohmh", "action",
]

inspections = inspections_raw[INSPECTION_COLS].copy()
print(f"inspections: {len(inspections):,} rows, {mb(inspections):.1f} MB")
inspections.head(3)

inspections: 93,106 rows, 45.2 MB


,inspection_id,camis,inspection_date,inspection_type,inspection_category,inspection_stage,score,grade,grade_date,is_gradeable,n_violations,n_critical,closed_by_dohmh,action
0,1,30075445,2023-08-01,Cycle Inspection / Initial Inspection,Cycle Inspection,Initial Inspection,38.0,NaN,NaN,True,3,2,False,Violations were cited in the following area(s).
1,2,30075445,2023-08-22,Cycle Inspection / Re-inspection,Cycle Inspection,Re-inspection,12.0,A,2023-08-22,True,3,1,False,Violations were cited in the following area(s).
2,3,30075445,2024-11-08,Cycle Inspection / Initial Inspection,Cycle Inspection,Initial Inspection,10.0,A,2024-11-08,True,3,1,False,Violations were cited in the following area(s).


### 2.3 `violations` — 288,486 rows, down to four columns

Everything else was a copy of something already stored elsewhere:

| Dropped column | Reachable instead by |
|---|---|
| `dba`, `boro`, `cuisine` | `violations -> inspections -> restaurants` |
| `inspection_date`, `inspection_type`, `inspection_category` | `violations -> inspections` |
| `score`, `grade`, `action` | `violations -> inspections` |
| `violation_description`, `critical_flag` | `violations -> violation_codes` |

In [7]:
n_before = len(violations_raw)

violations = (
    violations_raw
    .merge(violation_codes, on=["violation_code", "violation_description"], how="left")
    [["inspection_id", "violation_id", "is_critical"]]
    .reset_index(drop=True)
)
violations.insert(0, "violation_row_id", np.arange(1, len(violations) + 1))

assert len(violations) == n_before, "the lookup join changed the row count"
assert violations["violation_id"].notna().all(), "some violations matched no code"

print(f"violations: {len(violations):,} rows, {mb(violations):.1f} MB "
      f"(was {before['violations']:.1f} MB)")
print(f"Shrunk by {before['violations'] / mb(violations):.0f}x")
violations.head(3)

violations: 288,486 rows, 6.9 MB (was 251.8 MB)
Shrunk by 37x


,violation_row_id,inspection_id,violation_id,is_critical
0,1,7882,108,False
1,2,89630,4,True
2,3,27169,4,True


## 3. Referential integrity

Every foreign key must point at a row that exists. A database will enforce this on load — better to find any violation here than in a failed import.

In [8]:
checks = []

def check(name, condition, detail=""):
    checks.append({"check": name, "result": "PASS" if condition else "FAIL", "detail": detail})

check("restaurants.camis is unique", restaurants["camis"].is_unique)
check("inspections.inspection_id is unique", inspections["inspection_id"].is_unique)
check("violations.violation_row_id is unique", violations["violation_row_id"].is_unique)
check("violation_codes.violation_id is unique", violation_codes["violation_id"].is_unique)
check("neighborhoods.nta_code is unique", neighborhoods["nta_code"].is_unique)

check("inspections.camis -> restaurants",
      inspections["camis"].isin(restaurants["camis"]).all())
check("violations.inspection_id -> inspections",
      violations["inspection_id"].isin(inspections["inspection_id"]).all())
check("violations.violation_id -> violation_codes",
      violations["violation_id"].isin(violation_codes["violation_id"]).all())

# nta_code is nullable: 805 restaurants were never geocoded (notebook 01, issue M-04)
known_nta = restaurants["nta_code"].dropna()
check("restaurants.nta_code -> neighborhoods (where present)",
      known_nta.isin(neighborhoods["nta_code"]).all(),
      f"{restaurants['nta_code'].isna().sum()} rows have no NTA and stay NULL")

check("no rows lost", len(violations) == len(violations_raw)
      and len(inspections) == len(inspections_raw)
      and len(restaurants) == len(restaurants_raw))

result = pd.DataFrame(checks)
display(result)
assert (result["result"] == "PASS").all(), "referential integrity check failed"
print("\nAll referential integrity checks passed.")

,check,result,detail
0,restaurants.camis is unique,PASS,
1,inspections.inspection_id is unique,PASS,
2,violations.violation_row_id is unique,PASS,
3,violation_codes.violation_id is unique,PASS,
4,neighborhoods.nta_code is unique,PASS,
5,inspections.camis -> restaurants,PASS,
6,violations.inspection_id -> inspections,PASS,
7,violations.violation_id -> violation_codes,PASS,
8,restaurants.nta_code -> neighborhoods (where p...,PASS,805 rows have no NTA and stay NULL
9,no rows lost,PASS,



All referential integrity checks passed.


## 4. What normalisation bought

In [9]:
TABLES = {
    "restaurants": restaurants,
    "inspections": inspections,
    "violations": violations,
    "violation_codes": violation_codes,
    "neighborhoods": neighborhoods,
}

after = pd.DataFrame([
    {"table": name, "rows": len(t), "columns": t.shape[1], "size_mb": round(mb(t), 1)}
    for name, t in TABLES.items()
])
display(after)

total_before = sum(before.values())
total_after = after["size_mb"].sum()
print(f"Analysis tables:   {total_before:6.1f} MB")
print(f"Database tables:   {total_after:6.1f} MB")
print(f"Reduction:         {total_before / total_after:.1f}x")
print(f"\nSupabase free tier is 500 MB, so this fits with room to spare.")

,table,rows,columns,size_mb
0,restaurants,31222,23,19.5
1,inspections,93106,14,45.2
2,violations,288486,4,6.9
3,violation_codes,241,3,0.1
4,neighborhoods,195,4,0.0


Analysis tables:    321.9 MB
Database tables:     71.7 MB
Reduction:         4.5x

Supabase free tier is 500 MB, so this fits with room to spare.


## 5. The schema definition

Below is the SQL that creates these tables in Postgres. Three things it does that a CSV
cannot:

1. **Types** — a date is a date, not the text `"03/25/2024"`, so date arithmetic works
2. **Primary keys** — the database refuses to store two restaurants with the same `camis`
3. **Foreign keys** — the database refuses to store a violation pointing at an inspection
   that does not exist

The last two are the real value: rules that were assertions in a notebook become rules the
database itself enforces, on every write, forever.

The script also turns on **row level security**. Supabase publishes every table in the
`public` schema as a REST API, and the key the dashboard uses is shipped inside the app
where anyone can read it. Left open, that key would allow deleting the data. Enabling RLS
denies everything by default, and the policies then grant a single capability — anonymous
`select`. No insert, update or delete policy exists, so the API is read-only.

RLS and table privileges are two separate gates, and both must be open. A policy says which
*rows* a role may see; a `grant` says whether the role may address the table at all. The
script issues `grant select` and nothing more, so the anon role can read every row and
change none.

In [10]:
SCHEMA = """-- NYC Restaurant Intelligence Platform — database schema
-- Generated by notebooks/04_database_schema.ipynb
-- Target: PostgreSQL (Supabase)

-- Order matters: referenced tables must exist before the tables that point at them.
drop table if exists violations cascade;
drop table if exists inspections cascade;
drop table if exists restaurants cascade;
drop table if exists violation_codes cascade;
drop table if exists neighborhoods cascade;

-- ---------- reference tables ----------

create table neighborhoods (
    nta_code         text primary key,
    nta_name         text not null,
    borough          text not null,
    population_2010  integer
);

create table violation_codes (
    violation_id           integer primary key,
    violation_code         text not null,
    violation_description  text
);

-- ---------- restaurants ----------

create table restaurants (
    camis             text primary key,
    dba               text,
    boro              text,
    building          text,
    street            text,
    zipcode           text,
    phone             text,
    cuisine           text,
    latitude          double precision,
    longitude         double precision,
    nta_code          text references neighborhoods (nta_code),
    community_board   text,
    council_district  text,
    never_inspected   boolean,
    n_inspections     integer,
    first_inspection  date,
    last_inspection   date,
    avg_score         numeric(6, 2),
    latest_score      numeric(6, 2),
    latest_grade      text,
    total_violations  integer,
    total_critical    integer,
    ever_closed       boolean
);

-- ---------- inspections ----------

create table inspections (
    inspection_id        integer primary key,
    camis                text not null references restaurants (camis),
    inspection_date      date not null,
    inspection_type      text,
    inspection_category  text,
    inspection_stage     text,
    score                numeric(6, 2),
    grade                text,
    grade_date           date,
    is_gradeable         boolean,
    n_violations         integer,
    n_critical           integer,
    closed_by_dohmh      boolean,
    action               text
);

-- ---------- violations ----------

create table violations (
    violation_row_id  integer primary key,
    inspection_id     integer not null references inspections (inspection_id),
    violation_id      integer not null references violation_codes (violation_id),
    is_critical       boolean
);

-- ---------- indexes ----------
-- Foreign keys are not indexed automatically in Postgres. Without these, every
-- join and every dashboard filter scans the whole table.

create index idx_restaurants_boro     on restaurants (boro);
create index idx_restaurants_cuisine  on restaurants (cuisine);
create index idx_restaurants_nta      on restaurants (nta_code);
create index idx_inspections_camis    on inspections (camis);
create index idx_inspections_date     on inspections (inspection_date);
create index idx_violations_insp      on violations (inspection_id);
create index idx_violations_code      on violations (violation_id);

-- ---------- access control ----------
-- Supabase exposes every table in the public schema over a REST API. Without row
-- level security, anyone holding the (publicly shipped) anon key could delete the
-- data. Enabling RLS denies everything by default; the policies below then grant
-- exactly one capability: anonymous reads. There is no insert, update or delete
-- policy, so the API cannot write at all.

alter table neighborhoods       enable row level security;
alter table violation_codes     enable row level security;
alter table restaurants         enable row level security;
alter table inspections         enable row level security;
alter table violations          enable row level security;

create policy "public read" on neighborhoods       for select to anon using (true);
create policy "public read" on violation_codes     for select to anon using (true);
create policy "public read" on restaurants         for select to anon using (true);
create policy "public read" on inspections         for select to anon using (true);
create policy "public read" on violations          for select to anon using (true);

-- RLS decides which ROWS a role may see. Table privileges decide whether the role
-- may touch the table at all, and the two are independent — a table with a
-- permissive policy but no grant is still invisible to the API. Granting select
-- and nothing else means the anon role can read every row and modify none.
grant usage on schema public to anon;
grant select on all tables in schema public to anon;
"""

(SQL / "schema.sql").write_text(SCHEMA, encoding="utf-8")
print(f"Written: {SQL / 'schema.sql'}  ({len(SCHEMA.splitlines())} lines)")
print("\nPaste this into the Supabase SQL editor to create the tables.")

Written: /Users/qianyiyou/Desktop/gateway-restaurant-project/sql/schema.sql  (119 lines)

Paste this into the Supabase SQL editor to create the tables.


## 6. Export

CSV, because that is what Supabase's importer accepts. Postgres is stricter than pandas
about what it will read back, so three conversions happen here:

| Written as | Why |
|---|---|
| `2024-03-25` | Postgres `date` wants ISO order, not `03/25/2024` |
| `true` / `false` | Postgres `boolean` does not accept Python's `True` / `False` |
| `4`, not `4.0` | see below |

The last one is the subtle one. A pandas column containing any null is stored as float, so
integer counts come out as `4.0`. Postgres rejects that for an `integer` column outright —
and, more dangerously, **accepts it into a `text` column**, which would have quietly turned
zip code `10462` into the string `"10462.0"`. Casting to pandas' nullable `Int64` keeps the
nulls and drops the decimal point.

In [11]:
DATE_COLS = {
    "restaurants": ["first_inspection", "last_inspection"],
    "inspections": ["inspection_date", "grade_date"],
}


def whole_number_columns(df: pd.DataFrame) -> list[str]:
    """Float columns whose values are all whole numbers.

    Any pandas column containing a null becomes float64, so counts like
    n_inspections arrive here as 4.0 rather than 4. Written to CSV that way,
    Postgres rejects them for an integer column ("invalid input syntax for
    type integer: 4.0") and, worse, silently accepts them into a text column —
    turning zip code 10462 into the string "10462.0". Pandas' nullable Int64
    keeps the null and drops the decimal.
    """
    cols = []
    for col in df.columns:
        if pd.api.types.is_float_dtype(df[col]):
            values = df[col].dropna()
            if len(values) and (values % 1 == 0).all():
                cols.append(col)
    return cols


for name, table in TABLES.items():
    out = table.copy()

    for col in whole_number_columns(out):
        out[col] = out[col].astype("Int64")

    for col in DATE_COLS.get(name, []):
        out[col] = pd.to_datetime(out[col]).dt.strftime("%Y-%m-%d")

    for col in out.columns:
        if out[col].dtype == bool:
            out[col] = out[col].map({True: "true", False: "false"})

    out.to_csv(PROCESSED / f"{name}.csv", index=False)

print("Exported to data/processed/:\n")
for f in sorted(PROCESSED.glob("*.csv")):
    size = f.stat().st_size / 1024**2
    note = "  <- too large for the web importer, load with a script" if size > 40 else ""
    print(f"  {f.name:26} {size:6.1f} MB{note}")

Exported to data/processed/:

  borough_boundaries.csv        2.9 MB
  inspections.csv              15.8 MB
  neighborhoods.csv             0.0 MB
  restaurants.csv               5.0 MB
  violation_codes.csv           0.0 MB
  violations.csv                5.9 MB


## 7. Next: creating and loading the database

1. Create a Supabase project
2. Open the SQL editor and run `sql/schema.sql` — this creates the six empty tables
3. Load the CSVs from `data/processed/`, **in this order**, so foreign keys always point at
   rows that already exist:

   `neighborhoods` -> `violation_codes` -> `restaurants` -> `inspections` -> `violations`

   Load them out of order and Postgres will reject the rows — which is the foreign key
   constraint doing exactly what it was defined to do.

Normalisation made this easy: the largest file is now `inspections.csv` at 16 MB and the
whole set is about 30 MB, so every table can go through the browser importer. Had
`violations` stayed at its original 103 MB, it would have needed a scripted load.